In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
ENRICHMENT_DIR = PROJECT_ROOT / "data" / "enrichment"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw files:")

for path in RAW_DIR.iterdir():
    print("-", path.name)

Project root: /Users/mac/Ethiopia-FI-Interim
Raw files:
- reference_codes.csv
- Additional Data Points Guide.xlsx
- dataset_README.md
- ethiopia_fi_unified_data.xlsx


In [2]:
main_path = RAW_DIR / "ethiopia_fi_unified_data.xlsx"
reference_path = RAW_DIR / "reference_codes.csv"
guide_path = RAW_DIR / "Additional Data Points Guide.xlsx"

required_files = [main_path, reference_path, guide_path]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

workbook_sheets = pd.read_excel(main_path, sheet_name=None)

print("Main workbook sheets:", list(workbook_sheets.keys()))

main_sheet_name = next(
    name for name in workbook_sheets
    if "impact" not in name.lower()
)

impact_sheet_name = next(
    name for name in workbook_sheets
    if "impact" in name.lower()
)

main_data = workbook_sheets[main_sheet_name].copy()
impact_links = workbook_sheets[impact_sheet_name].copy()

unified = pd.concat(
    [main_data, impact_links],
    ignore_index=True,
    sort=False
)

reference_codes = pd.read_csv(reference_path)
guide_sheets = pd.read_excel(guide_path, sheet_name=None)

print("Main sheet:", main_sheet_name, main_data.shape)
print("Impact sheet:", impact_sheet_name, impact_links.shape)
print("Combined dataset:", unified.shape)
print("Reference codes:", reference_codes.shape)
print("Guide sheets:", list(guide_sheets.keys()))

display(unified.head())
display(reference_codes.head())

Main workbook sheets: ['ethiopia_fi_unified_data', 'Impact_sheet']
Main sheet: ethiopia_fi_unified_data (43, 34)
Impact sheet: Impact_sheet (14, 35)
Combined dataset: (57, 35)
Reference codes: (71, 4)
Guide sheets: ['A. Alternative Baselines', 'B. Direct Corrln', 'C. Indirect Corrln', 'D. Market Naunces']


,record_id,record_type,category,pillar,indicator,indicator_code,indicator_direction,value_numeric,value_text,value_type,unit,observation_date,period_start,period_end,fiscal_year,gender,location,region,source_name,source_type,source_url,confidence,related_indicator,relationship_type,impact_direction,impact_magnitude,impact_estimate,lag_months,evidence_basis,comparable_country,collected_by,collection_date,original_text,notes,parent_id
0,REC_0001,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,22.0,NaN,percentage,%,2014-12-31,NaT,NaT,2014,all,national,NaN,Global Findex 2014,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20 00:00:00,NaN,Baseline year,NaN,NaN
1,REC_0002,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,35.0,NaN,percentage,%,2017-12-31,NaT,NaT,2017,all,national,NaN,Global Findex 2017,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20 00:00:00,NaN,NaN,NaN,NaN
2,REC_0003,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,46.0,NaN,percentage,%,2021-12-31,NaT,NaT,2021,all,national,NaN,Global Findex 2021,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20 00:00:00,NaN,NaN,NaN,NaN
3,REC_0004,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,56.0,NaN,percentage,%,2021-12-31,NaT,NaT,2021,male,national,NaN,Global Findex 2021,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20 00:00:00,NaN,Gender disaggregated,NaN,NaN
4,REC_0005,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,36.0,NaN,percentage,%,2021-12-31,NaT,NaT,2021,female,national,NaN,Global Findex 2021,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20 00:00:00,NaN,Gender disaggregated,NaN,NaN


,field,code,description,applies_to
0,record_type,observation,Actual measured value from a source,All
1,record_type,event,Policy launch market event or milestone,All
2,record_type,impact_link,Relationship between event and indicator (link...,All
3,record_type,target,Policy target or official goal,All
4,record_type,baseline,Starting point for comparison,All


In [3]:
summary_columns = [
    "record_type",
    "pillar",
    "source_type",
    "confidence"
]

for column in summary_columns:
    if column in unified.columns:
        print(f"\nCounts by {column}")

        summary = (
            unified[column]
            .fillna("Missing")
            .value_counts(dropna=False)
            .rename_axis(column)
            .reset_index(name="count")
        )

        display(summary)
    else:
        print(f"Column not found: {column}")


Counts by record_type


,record_type,count
0,observation,30
1,impact_link,14
2,event,10
3,target,3



Counts by pillar


,pillar,count
0,ACCESS,20
1,USAGE,17
2,Missing,10
3,GENDER,6
4,AFFORDABILITY,4



Counts by source_type


,source_type,count
0,operator,15
1,Missing,14
2,survey,10
3,regulator,7
4,research,4
5,policy,3
6,calculated,2
7,news,2



Counts by confidence


,confidence,count
0,high,44
1,medium,13


In [4]:
date_columns = ["observation_date", "event_date"]

for column in date_columns:
    if column in unified.columns:
        unified[column] = pd.to_datetime(
            unified[column],
            errors="coerce"
        )

observations = unified[
    unified["record_type"].astype(str).str.lower().eq("observation")
].copy()

if "observation_date" in observations.columns:
    valid_dates = observations["observation_date"].dropna()

    print(
        "Observation date range:",
        valid_dates.min(),
        "to",
        valid_dates.max()
    )

if "indicator_code" in observations.columns:
    indicator_coverage = (
        observations
        .groupby("indicator_code", dropna=False)
        .agg(
            number_of_records=("indicator_code", "size"),
            first_date=("observation_date", "min"),
            last_date=("observation_date", "max")
        )
        .reset_index()
        .sort_values(
            "number_of_records",
            ascending=False
        )
    )

    display(indicator_coverage)

Observation date range: 2014-12-31 00:00:00 to 2025-12-31 00:00:00


,indicator_code,number_of_records,first_date,last_date
4,ACC_OWNERSHIP,6,2014-12-31,2024-11-29
1,ACC_FAYDA,3,2024-08-15,2025-05-15
0,ACC_4G_COV,2,2023-06-30,2025-06-30
2,ACC_MM_ACCOUNT,2,2021-12-31,2024-11-29
6,GEN_GAP_ACC,2,2021-12-31,2024-11-29
15,USG_P2P_COUNT,2,2024-07-07,2025-07-07
12,USG_CROSSOVER,1,2025-07-07,2025-07-07
17,USG_TELEBIRR_USERS,1,2025-06-30,2025-06-30
16,USG_P2P_VALUE,1,2025-07-07,2025-07-07
14,USG_MPESA_USERS,1,2024-12-31,2024-12-31


In [5]:
events = unified[
    unified["record_type"].astype(str).str.lower().eq("event")
].copy()

links = unified[
    unified["record_type"].astype(str).str.lower().eq("impact_link")
].copy()

print("Number of events:", len(events))
print("Number of impact links:", len(links))

event_columns = [
    column for column in [
        "record_id",
        "indicator",
        "indicator_code",
        "category",
        "event_date",
        "source_name"
    ]
    if column in events.columns
]

link_columns = [
    column for column in [
        "record_id",
        "parent_id",
        "pillar",
        "related_indicator",
        "impact_direction",
        "impact_magnitude",
        "lag_months",
        "evidence_basis"
    ]
    if column in links.columns
]

display(events[event_columns])
display(links[link_columns])

if "parent_id" in links.columns and "record_id" in events.columns:
    event_link_summary = links.merge(
        events[event_columns],
        left_on="parent_id",
        right_on="record_id",
        how="left",
        suffixes=("_link", "_event")
    )

    display(event_link_summary)

Number of events: 10
Number of impact links: 14


,record_id,indicator,indicator_code,category,source_name
33,EVT_0001,Telebirr Launch,EVT_TELEBIRR,product_launch,Ethio Telecom
34,EVT_0002,Safaricom Ethiopia Commercial Launch,EVT_SAFARICOM,market_entry,News
35,EVT_0003,M-Pesa Ethiopia Launch,EVT_MPESA,product_launch,Safaricom
36,EVT_0004,Fayda Digital ID Program Rollout,EVT_FAYDA,infrastructure,NIDP
37,EVT_0005,Foreign Exchange Liberalization,EVT_FX_REFORM,policy,NBE
38,EVT_0006,P2P Transaction Count Surpasses ATM,EVT_CROSSOVER,milestone,EthSwitch
39,EVT_0007,M-Pesa EthSwitch Integration,EVT_MPESA_INTEROP,partnership,EthSwitch
40,EVT_0008,EthioPay Instant Payment System Launch,EVT_ETHIOPAY,infrastructure,NBE/EthSwitch
41,EVT_0009,NFIS-II Strategy Launch,EVT_NFIS2,policy,NBE
42,EVT_0010,Safaricom Ethiopia Price Increase,EVT_SAFCOM_PRICE,pricing,News


,record_id,parent_id,pillar,related_indicator,impact_direction,impact_magnitude,lag_months,evidence_basis
43,IMP_0001,EVT_0001,ACCESS,ACC_OWNERSHIP,increase,high,12.0,literature
44,IMP_0002,EVT_0001,USAGE,USG_TELEBIRR_USERS,increase,high,3.0,empirical
45,IMP_0003,EVT_0001,USAGE,USG_P2P_COUNT,increase,high,6.0,empirical
46,IMP_0004,EVT_0002,ACCESS,ACC_4G_COV,increase,medium,12.0,empirical
47,IMP_0005,EVT_0002,AFFORDABILITY,AFF_DATA_INCOME,decrease,medium,12.0,literature
48,IMP_0006,EVT_0003,USAGE,USG_MPESA_USERS,increase,high,3.0,empirical
49,IMP_0007,EVT_0003,ACCESS,ACC_MM_ACCOUNT,increase,medium,6.0,theoretical
50,IMP_0008,EVT_0004,ACCESS,ACC_OWNERSHIP,increase,medium,24.0,literature
51,IMP_0009,EVT_0004,GENDER,GEN_GAP_ACC,decrease,medium,24.0,literature
52,IMP_0010,EVT_0005,AFFORDABILITY,AFF_DATA_INCOME,increase,high,3.0,empirical


,record_id_link,record_type,category_link,pillar,indicator_link,indicator_code_link,indicator_direction,value_numeric,value_text,value_type,unit,observation_date,period_start,period_end,fiscal_year,gender,location,region,source_name_link,source_type,source_url,confidence,related_indicator,relationship_type,impact_direction,impact_magnitude,impact_estimate,lag_months,evidence_basis,comparable_country,collected_by,collection_date,original_text,notes,parent_id,record_id_event,indicator_event,indicator_code_event,category_event,source_name_event
0,IMP_0001,impact_link,NaN,ACCESS,Telebirr effect on Account Ownership,NaN,NaN,15.0,NaN,percentage,%,2021-05-17,NaN,NaN,NaN,all,national,NaN,NaN,NaN,NaN,medium,ACC_OWNERSHIP,direct,increase,high,15.0,12.0,literature,Kenya,Example_Trainee,2025-01-20 00:00:00,NaN,Kenya M-Pesa showed +20pp over 5 years,EVT_0001,EVT_0001,Telebirr Launch,EVT_TELEBIRR,product_launch,Ethio Telecom
1,IMP_0002,impact_link,NaN,USAGE,Telebirr effect on Telebirr Users,NaN,NaN,NaN,NaN,count,users,2021-05-17,NaN,NaN,NaN,all,national,NaN,NaN,NaN,NaN,high,USG_TELEBIRR_USERS,direct,increase,high,NaN,3.0,empirical,NaN,Example_Trainee,2025-01-20 00:00:00,NaN,Direct subscriber acquisition,EVT_0001,EVT_0001,Telebirr Launch,EVT_TELEBIRR,product_launch,Ethio Telecom
2,IMP_0003,impact_link,NaN,USAGE,Telebirr effect on P2P Transactions,NaN,NaN,25.0,NaN,percentage,%,2021-05-17,NaN,NaN,NaN,all,national,NaN,NaN,NaN,NaN,medium,USG_P2P_COUNT,direct,increase,high,25.0,6.0,empirical,NaN,Example_Trainee,2025-01-20 00:00:00,NaN,New digital payment channel,EVT_0001,EVT_0001,Telebirr Launch,EVT_TELEBIRR,product_launch,Ethio Telecom
3,IMP_0004,impact_link,NaN,ACCESS,Safaricom effect on 4G Coverage,NaN,NaN,15.0,NaN,percentage,%,2022-08-01,NaN,NaN,NaN,all,national,NaN,NaN,NaN,NaN,medium,ACC_4G_COV,direct,increase,medium,15.0,12.0,empirical,NaN,Example_Trainee,2025-01-20 00:00:00,NaN,Network investment from competition,EVT_0002,EVT_0002,Safaricom Ethiopia Commercial Launch,EVT_SAFARICOM,market_entry,News
4,IMP_0005,impact_link,NaN,AFFORDABILITY,Safaricom effect on Data Affordability,NaN,NaN,-20.0,NaN,percentage,%,2022-08-01,NaN,NaN,NaN,all,national,NaN,NaN,NaN,NaN,medium,AFF_DATA_INCOME,indirect,decrease,medium,-20.0,12.0,literature,Rwanda,Example_Trainee,2025-01-20 00:00:00,NaN,Competition typically reduces prices,EVT_0002,EVT_0002,Safaricom Ethiopia Commercial Launch,EVT_SAFARICOM,market_entry,News
5,IMP_0006,impact_link,NaN,USAGE,M-Pesa effect on M-Pesa Users,NaN,NaN,NaN,NaN,count,users,2023-08-01,NaN,NaN,NaN,all,national,NaN,NaN,NaN,NaN,high,USG_MPESA_USERS,direct,increase,high,NaN,3.0,empirical,NaN,Example_Trainee,2025-01-20 00:00:00,NaN,Direct subscriber acquisition,EVT_0003,EVT_0003,M-Pesa Ethiopia Launch,EVT_MPESA,product_launch,Safaricom
6,IMP_0007,impact_link,NaN,ACCESS,M-Pesa effect on Mobile Money Account Rate,NaN,NaN,5.0,NaN,percentage,%,2023-08-01,NaN,NaN,NaN,all,national,NaN,NaN,NaN,NaN,medium,ACC_MM_ACCOUNT,direct,increase,medium,5.0,6.0,theoretical,NaN,Example_Trainee,2025-01-20 00:00:00,NaN,Second provider adds incremental accounts,EVT_0003,EVT_0003,M-Pesa Ethiopia Launch,EVT_MPESA,product_launch,Safaricom
7,IMP_0008,impact_link,NaN,ACCESS,Fayda effect on Account Ownership,NaN,NaN,10.0,NaN,percentage,%,2024-01-01,NaN,NaN,NaN,all,national,NaN,NaN,NaN,NaN,medium,ACC_OWNERSHIP,enabling,increase,medium,10.0,24.0,literature,India,Example_Trainee,2025-01-20 00:00:00,NaN,Aadhaar enabled +15-20% account opening in India,EVT_0004,EVT_0004,Fayda Digital ID Program Rollout,EVT_FAYDA,infrastructure,NIDP
8,IMP_0009,impact_link,NaN,GENDER,Fayda effect on Gender Gap,NaN,NaN,-5.0,NaN,gap_pp,pp,2024-01-01,NaN,NaN,NaN,all,national,NaN,NaN,NaN,NaN,medium,GEN_GAP_ACC,indirect,decrease,medium,-5.0,24.0,literature,India,Example_Trainee,2025-01-20 00:00:00,NaN,Women disproportionately lack traditional ID,EVT_0004,EVT_0004,Fayda Digital ID Program Rollout,EVT_FAYDA,infrastructure,NIDP
9,IMP_0010,impact_link,NaN,AFFORDABILITY,FX Reform effect on Data A

In [6]:
missing_values = (
    unified
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename_axis("column")
    .reset_index(name="missing_percent")
)

display(missing_values)

print("Fully duplicated rows:", unified.duplicated().sum())

if "record_id" in unified.columns:
    print(
        "Duplicated record IDs:",
        unified["record_id"].duplicated().sum()
    )

if "value_numeric" in unified.columns:
    unified["value_numeric"] = pd.to_numeric(
        unified["value_numeric"],
        errors="coerce"
    )

    display(unified["value_numeric"].describe())

,column,missing_percent
0,region,100.000000
1,period_end,82.456140
2,category,82.456140
3,value_text,82.456140
4,period_start,82.456140
5,impact_estimate,78.947368
6,notes,75.438596
7,evidence_basis,75.438596
8,lag_months,75.438596
9,impact_magnitude,75.438596


Fully duplicated rows: 0
Duplicated record IDs: 0


count    4.500000e+01
mean     6.920656e+10
std      3.632860e+11
min     -2.000000e+01
25%      1.500000e+01
50%      3.600000e+01
75%      8.000000e+06
max      2.380000e+12
Name: value_numeric, dtype: float64

In [8]:
enrichment_path = ENRICHMENT_DIR / "enrichment_records.csv"

if not enrichment_path.exists():
    raise FileNotFoundError(
        f"Enrichment file not found: {enrichment_path}"
    )

enrichment = pd.read_csv(enrichment_path)

print("New enrichment records:", len(enrichment))
display(enrichment)

enrichment_aligned = enrichment.reindex(
    columns=unified.columns
)

enriched = pd.concat(
    [unified, enrichment_aligned],
    ignore_index=True,
    sort=False
)

output_path = (
    PROCESSED_DIR /
    "ethiopia_fi_enriched.csv"
)

enriched.to_csv(output_path, index=False)

print("Original combined records:", len(unified))
print("Added records:", len(enrichment))
print("Final enriched records:", len(enriched))
print("Saved to:", output_path)

New enrichment records: 5


,record_id,record_type,pillar,indicator,indicator_code,value_numeric,observation_date,event_date,category,parent_id,related_indicator,impact_direction,impact_magnitude,lag_months,evidence_basis,source_name,source_url,original_text,confidence,collected_by,collection_date,notes
0,ENR_OBS_001,observation,ACCESS,Telebirr Agent Network Size,ACC_TELEBIRR_AGENTS,320300.0,2025-06-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ethio Telecom 2024/25 Annual Performance Report,https://www.ethiotelecom.et/2024-25-fiscal-yea...,agents to 320.3 thousand,high,Rediet Shewarega,2026-07-20,Converted 320.3 thousand to 320300. Agent avai...
1,ENR_OBS_002,observation,USAGE,Telebirr Merchant Network Size,USG_TELEBIRR_MERCHANTS,310100.0,2025-06-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ethio Telecom 2024/25 Annual Performance Report,https://www.ethiotelecom.et/2024-25-fiscal-yea...,merchants to 310.1 thousand,high,Rediet Shewarega,2026-07-20,Converted 310.1 thousand to 310100. Merchant c...
2,ENR_OBS_003,observation,USAGE,Telebirr Digital Savings Customers,USG_TELEBIRR_SAVERS,1770000.0,2025-06-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ethio Telecom 2024/25 Annual Performance Report,https://www.ethiotelecom.et/2024-25-fiscal-yea...,1.77 million customers engaged in digital savings,high,Rediet Shewarega,2026-07-20,Measures deeper use of digital financial servi...
3,ENR_EVT_001,event,NaN,National Digital Payments Strategy Phase Two L...,EVT_NDPS2,NaN,NaN,2025-03-28,policy,NaN,NaN,NaN,NaN,NaN,NaN,National Bank of Ethiopia,https://nbe.gov.et/nbe_news/ethiopia-launches-...,announced the launch of Phase Two of the Natio...,high,Rediet Shewarega,2026-07-20,The event remains pillar-neutral. Its expected...
4,ENR_IMP_001,impact_link,USAGE,NaN,NaN,NaN,NaN,NaN,NaN,ENR_EVT_001,USG_DIGITAL_PAYMENT,increase,medium,12.0,theoretical,National Bank of Ethiopia,https://nbe.gov.et/nbe_news/ethiopia-launches-...,focus on deepening usage of digital payments,medium,Rediet Shewarega,2026-07-20,Assumes implementation and user adoption build...


Original combined records: 57
Added records: 5
Final enriched records: 62
Saved to: /Users/mac/Ethiopia-FI-Interim/data/processed/ethiopia_fi_enriched.csv


# Task 1 Findings

The supplied data contained 43 primary records and 14 impact-link
records, producing 57 combined starter records. Five additional records
were added, resulting in an enriched dataset containing 62 records.

The unified schema separates observations, events, targets, and impact
links. Events remain pillar-neutral because their effects may apply to
multiple financial-inclusion dimensions. These effects are represented
through impact-link records.

The main account-ownership series is sparse because Global Findex
observations are available only for selected survey years. This limits
the reliability of complex time-series models and requires explicit
uncertainty and scenario analysis.

Operator registrations and administrative counts should not be
interpreted as unique active adult users. Registered accounts can include
inactive accounts, duplicated users, and users who already own formal
bank accounts.

The enrichment added information about Telebirr agents, merchants,
digital-savings usage, and the National Digital Payments Strategy Phase
Two. These variables help represent infrastructure, merchant acceptance,
deeper financial-service usage, and policy interventions.